# 📋 Module 4.2: Registering Models

**Goal**: Learn two methods to register models in the MLFlow Model Registry.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)
mlflow.set_experiment("04_model_registry")

wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)

client = MlflowClient()
print("✅ Setup complete!")

## Method 1: Register During `log_model()`

The simplest way — add `registered_model_name` when logging the model.

In [ ]:
MODEL_NAME = "wine-quality-classifier"

with mlflow.start_run(run_name="register_during_log"):
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_params({"n_estimators": 100, "max_depth": 5})
    mlflow.log_metric("accuracy", accuracy)
    
    # Register model by adding registered_model_name!
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name=MODEL_NAME  # ← This registers it!
    )
    
    run_id = mlflow.active_run().info.run_id
    print(f"✅ Model registered as '{MODEL_NAME}'")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   Run ID: {run_id}")

## Method 2: Register After the Fact with `register_model()`

Train and log a model first, then register it separately. Useful when you want to review before registering.

In [ ]:
# Step 1: Train and log (without registering)
with mlflow.start_run(run_name="register_after_fact") as run:
    model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_params({"n_estimators": 150, "learning_rate": 0.1})
    mlflow.log_metric("accuracy", accuracy)
    
    # Log model WITHOUT registering
    mlflow.sklearn.log_model(model, artifact_path="model")
    
    run_id = run.info.run_id
    print(f"Model logged (not registered yet). Run ID: {run_id}")
    print(f"Accuracy: {accuracy:.4f}")

In [ ]:
# Step 2: Now register it (separate step)
result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name=MODEL_NAME
)

print(f"✅ Model registered!")
print(f"   Name: {result.name}")
print(f"   Version: {result.version}")
print(f"   Source: {result.source}")

## Viewing Registered Models

In [ ]:
# List all registered models
registered_models = client.search_registered_models()

print("📋 Registered Models:")
for rm in registered_models:
    print(f"\n  Model: {rm.name}")
    print(f"  Latest Versions:")
    for v in rm.latest_versions:
        print(f"    - Version {v.version} (status: {v.status})")

In [ ]:
# Get details of a specific registered model
model_details = client.get_registered_model(MODEL_NAME)

print(f"Model: {model_details.name}")
print(f"Description: {model_details.description or 'None'}")
print(f"Tags: {model_details.tags}")
print(f"\nVersions:")
for v in model_details.latest_versions:
    print(f"  v{v.version}: run_id={v.run_id[:8]}...")

## Adding Description and Tags to Registered Models

In [ ]:
# Add a description
client.update_registered_model(
    name=MODEL_NAME,
    description="Classifies wine samples into 3 quality categories based on chemical properties."
)

# Add tags
client.set_registered_model_tag(MODEL_NAME, "team", "ml-engineering")
client.set_registered_model_tag(MODEL_NAME, "task", "classification")
client.set_registered_model_tag(MODEL_NAME, "dataset", "wine-quality")

# Verify
model_details = client.get_registered_model(MODEL_NAME)
print(f"Description: {model_details.description}")
print(f"Tags: {model_details.tags}")

## 🔑 Key Takeaways

1. **Method 1** (`registered_model_name` in `log_model`): Quick, one-step registration
2. **Method 2** (`register_model` after logging): More control, review before registering
3. Each registration creates a new **version** automatically
4. Use **descriptions** and **tags** to document your models
5. The **MlflowClient** gives you full programmatic access to the registry

---

## ➡️ Next: `03_model_versions.ipynb` — Managing model versions